In [3]:
import pandas as pd 

df = pd.read_csv("sklearn_data.csv")

df

,tag,text
0,h1,User Guide #
1,p,Section Navigation
2,p,previous
3,p,Installing scikit-learn
4,p,next
...,...,...
12433,pre,>>> from sklearn.datasets import load_iris >>>...
12434,pre,">>> from sklearn.metrics import make_scorer , ..."
12435,h1,sklearn.ensemble #
12436,p,"Ensemble-based methods for classification, reg..."


In [4]:
df.drop(columns=['tag'],inplace=True)
df.set_index("text")

""
text
User Guide #
Section Navigation
previous
Installing scikit-learn
next
...
">>> from sklearn.datasets import load_iris >>> from sklearn.metrics import check_scoring >>> from sklearn.tree import DecisionTreeClassifier >>> X , y = load_iris ( return_X_y = True ) >>> classifier = DecisionTreeClassifier ( max_depth = 2 ) . fit ( X , y ) >>> scorer = check_scoring ( classifier , scoring = 'accuracy' ) >>> scorer ( classifier , X , y ) 0.96..."
">>> from sklearn.metrics import make_scorer , accuracy_score , mean_squared_log_error >>> X , y = load_iris ( return_X_y = True ) >>> y *= - 1 >>> clf = DecisionTreeClassifier () . fit ( X , y ) >>> scoring = { ... ""accuracy"" : make_scorer ( accuracy_score ), ... ""mean_squared_log_error"" : make_scorer ( mean_squared_log_error ), ... } >>> scoring_call = check_scoring ( estimator = clf , scoring = scoring , raise_exc = False ) >>> scores = scoring_call ( clf , X , y ) >>> scores {'accuracy': 1.0, 'mean_squared_log_error': 'Traceback ...'}"
sklearn.ensemble #


In [5]:
def chunking(data: pd.DataFrame, chunk_size=500):
    chunks = []

    for i in range(0, len(data), chunk_size):
        chunks.append(data.iloc[i:i + chunk_size])

    return chunks

In [6]:
chunking(df)

[                                                  text
 0                                         User Guide #
 1                                   Section Navigation
 2                                             previous
 3                              Installing scikit-learn
 4                                                 next
 ..                                                 ...
 495  Computing Euclidean distances in this d-dimens...
 496  We can reduce the dimension even more, to a ch...
 497  Shrinkage is a form of regularization used to ...
 498  The shrinkage parameter can also be manually s...
 499  The shrunk Ledoit and Wolf estimator of covari...
 
 [500 rows x 1 columns],
                                                   text
 500  The covariance estimator can be chosen using t...
 501  Normal, Ledoit-Wolf and OAS Linear Discriminan...
 502  Using LDA and QDA requires computing the log-p...
 503  The ‘svd’ solver is the default solver used fo...
 504  The 'lsqr' solv

john


In [7]:
chunks = chunking(df)

In [8]:
chunk_texts = []

for chunk in chunks:
    text = " ".join(chunk["text"].astype(str).tolist())
    chunk_texts.append(text)

In [33]:
from langchain_core.documents import Document

docs = [Document(page_content=text) for text in chunk_texts]

In [34]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [35]:
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

vdb = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="route"
)

In [36]:
retriever = vdb.as_retriever(search_kwargs={"k":10})

In [38]:
query = "what is supervised learining"

In [39]:
results = vdb.similarity_search_with_score(query, k=3)

In [43]:
context = "\n\n".join(
    [doc.page_content for doc, score in results]
)

In [44]:
context

'Examples concerning the sklearn.gaussian_process module. Gaussian process classification (GPC) on iris dataset Gaussian processes on discrete data structures Illustration of Gaussian process classification (GPC) on the XOR dataset Illustration of prior and posterior Gaussian process for different kernels Iso-probability lines for Gaussian Processes classification (GPC) Probabilistic predictions with Gaussian process classification (GPC) Examples concerning the sklearn.linear_model module. Lasso on dense and sparse data Examples related to the sklearn.inspection module. Failure of Machine Learning to infer causal effects Examples concerning the sklearn.kernel_approximation module. Examples concerning the sklearn.manifold module. Comparison of Manifold Learning methods Manifold Learning methods on a severed sphere Swiss Roll And Swiss-Hole Reduction t-SNE: The effect of various perplexity values on the shape Miscellaneous and introductory examples for scikit-learn. Comparing anomaly det

In [45]:
prompt = f"""
You are a Retrieval-Augmented Generation (RAG) assistant.

Answer ONLY using the context below.

If the answer cannot be found in the context,
reply exactly:

"I cannot find the answer in the provided context."

Context:
{context}

Question:
{query}

Answer:
"""

In [ ]:
import google.generativeai as genai

genai.configure(api_key="private")

model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content(prompt)

print(response.text)

I cannot find the answer in the provided context.
